# Module 07 — Optimizers: SGD, Momentum, Adam, and AdamW

This notebook starts with scalar update rules, simulates Adam’s moment estimates, compares four optimizers on one regression task, and inspects optimizer state.

## 1. SGD and momentum by hand

The momentum buffer combines the current gradient with recent history.

In [ ]:
def momentum_updates(gradients, learning_rate=0.1, beta=0.9):
    buffer = 0.0
    rows = []

    for step, gradient in enumerate(gradients, start=1):
        buffer = beta * buffer + gradient
        update = -learning_rate * buffer
        rows.append({
            "step": step,
            "gradient": gradient,
            "buffer": buffer,
            "update": update,
        })

    return rows


for row in momentum_updates([4, 4, 4, -4, -4, -4]):
    print(row)

## 2. Simulate Adam’s first and second moments

In [ ]:
import math


def simulate_adam(
    gradients,
    learning_rate=0.1,
    beta1=0.9,
    beta2=0.999,
    epsilon=1e-8,
):
    first_moment = 0.0
    second_moment = 0.0
    rows = []

    for step, gradient in enumerate(gradients, start=1):
        first_moment = beta1 * first_moment + (1 - beta1) * gradient
        second_moment = beta2 * second_moment + (1 - beta2) * gradient**2
        corrected_m = first_moment / (1 - beta1**step)
        corrected_v = second_moment / (1 - beta2**step)
        ratio = corrected_m / (math.sqrt(corrected_v) + epsilon)

        rows.append({
            "step": step,
            "gradient": gradient,
            "sgd_update": -learning_rate * gradient,
            "adam_update": -learning_rate * ratio,
            "ratio": ratio,
        })

    return rows


sequences = {
    "constant": [1, 1, 1, 1, 1, 1],
    "spike": [1, 2, 4, 8, 16, 32],
    "alternating": [10, -10, 10, -10, 10, -10],
}

for name, gradients in sequences.items():
    print(name)
    for row in simulate_adam(gradients):
        print(row)

## 3. Compare four optimizers on the same regression task

In [ ]:
import copy
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class TinyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.Tanh(),
            nn.Linear(32, 1),
        )

    def forward(self, inputs):
        return self.net(inputs)


x = torch.randn(256, 2, device=device)
y = 3 * x[:, 0:1] - 2 * x[:, 1:2]
reference_model = TinyModel().to(device)
initial_weights = copy.deepcopy(reference_model.state_dict())


def run_optimizer(name, lr=0.01, steps=200):
    model = TinyModel().to(device)
    model.load_state_dict(initial_weights)

    if name == "SGD":
        optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    elif name == "Momentum":
        optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    elif name == "Adam":
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0.0)
    elif name == "AdamW":
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.0)
    else:
        raise ValueError(f"Unknown optimizer: {name}")

    losses = []
    for _ in range(steps):
        optimizer.zero_grad(set_to_none=True)
        loss = F.mse_loss(model(x), y)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    return model, optimizer, losses


results = {
    name: run_optimizer(name)
    for name in ("SGD", "Momentum", "Adam", "AdamW")
}

plt.figure(figsize=(9, 5))
for name, (_, _, losses) in results.items():
    plt.plot(losses, label=name)

plt.yscale("log")
plt.xlabel("Optimizer step")
plt.ylabel("Training MSE")
plt.title("Same model, same learning rate, different optimizers")
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## 4. Inspect optimizer state and memory

Adam and AdamW lazily create moment tensors when a parameter receives its first update.

In [ ]:
model, optimizer, _ = run_optimizer("AdamW", lr=1e-3, steps=1)
first_parameter = next(model.parameters())
state = optimizer.state[first_parameter]

print("Parameter shape:", tuple(first_parameter.shape))
for key, value in state.items():
    if torch.is_tensor(value):
        print(key, tuple(value.shape))
    else:
        print(key, value)

## Exercises

1. Change the gradient sequence in the Adam simulator and inspect `m`, `v`, and the normalized ratio.
2. Tune the learning rate separately for SGD, Momentum, Adam, and AdamW.
3. Enable nonzero weight decay for Adam and AdamW and compare parameter norms.
4. Estimate optimizer-state memory for a larger parameter tensor.